In [0]:
import sys
sys.path.insert(0, '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src')

from bronze.config import YamlConfigStore

store = YamlConfigStore('/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/configs/example.yaml')
print('Jobs found:', [j.job_id for j in store.list_jobs()])
print('All good!')

Jobs found: ['customers_raw', 'orders_raw']
All good!


In [0]:
# Connect to your ADLS storage account
spark.conf.set(
    "fs.azure.account.key.medelliondataarcs.dfs.core.windows.net",
    "<your-storage-key>"
)

# Test the connection
files = dbutils.fs.ls("abfss://landing@medelliondataarcs.dfs.core.windows.net/")
for f in files:
    print(f.path)

abfss://landing@medelliondataarcs.dfs.core.windows.net/sample1.csv
abfss://landing@medelliondataarcs.dfs.core.windows.net/sample2.json
abfss://landing@medelliondataarcs.dfs.core.windows.net/sample3.csv


In [0]:
files = dbutils.fs.ls("abfss://landing@medelliondataarcs.dfs.core.windows.net/")
for f in files:
    print(f.path)

abfss://landing@medelliondataarcs.dfs.core.windows.net/sample1.csv
abfss://landing@medelliondataarcs.dfs.core.windows.net/sample2.json
abfss://landing@medelliondataarcs.dfs.core.windows.net/sample3.csv


In [0]:
import sys
sys.path.insert(0, '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src')

from bronze.metadata import attach_metadata, generate_run_id, generate_batch_id

# Generate IDs
run_id = generate_run_id()
batch_id = generate_batch_id("customers_raw", run_id)
print(f"run_id:   {run_id}")
print(f"batch_id: {batch_id}")

# Create a tiny test DataFrame
data = [("1", "Alice"), ("2", "Bob"), ("3", "Charlie")]
df = spark.createDataFrame(data, ["id", "name"])

# Attach metadata
df_with_meta = attach_metadata(
    df,
    batch_id=batch_id,
    run_id=run_id,
    source_file_name="sample1.csv",
    source_file_hash="abc123def456"
)

df_with_meta.show(truncate=False)
print("Metadata builder works!")

run_id:   26be9516-899b-4950-b6fa-7fbdda33e740
batch_id: customers_raw__26be9516
+---+-------+-----------------------+------------------------------------+-----------------+-----------------+-----------------------+---------------+-----------+------------------------------------+
|id |name   |_batch_id              |_run_id                             |_source_file_name|_source_file_hash|_ingestion_timestamp   |_ingestion_date|_record_seq|_source_row_uuid                    |
+---+-------+-----------------------+------------------------------------+-----------------+-----------------+-----------------------+---------------+-----------+------------------------------------+
|1  |Alice  |customers_raw__26be9516|26be9516-899b-4950-b6fa-7fbdda33e740|sample1.csv      |abc123def456     |2026-04-29 11:51:46.589|2026-04-29     |8589934592 |941861c1-603d-40f8-95c3-7194f77db742|
|2  |Bob    |customers_raw__26be9516|26be9516-899b-4950-b6fa-7fbdda33e740|sample1.csv      |abc123def456     |2026-04-2

In [0]:
import sys
mods_to_remove = [k for k in sys.modules if 'bronze' in k]
for m in mods_to_remove:
    del sys.modules[m]

sys.path.insert(0, '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src')

from bronze.readers.adls_reader import read_from_adls
from bronze.config import YamlConfigStore

spark.conf.set(
    "fs.azure.account.key.medelliondataarcs.dfs.core.windows.net",
    "<your-storage-key>"
)

store = YamlConfigStore('/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/configs/example.yaml')
job = store.get_job("customers_raw")

clean_df, corrupt_df = read_from_adls(
    spark,
    "abfss://landing@medelliondataarcs.dfs.core.windows.net/customers/sample1.csv",
    job
)

print("Clean rows:")
clean_df.show()
print(f"Corrupt rows: {corrupt_df.count()}")
print("ADLS reader works!")

Clean rows:
+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

Corrupt rows: 0
ADLS reader works!


In [0]:
with open('/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src/bronze/__init__.py', 'w') as f:
    f.write('# Bronze package\n')

print("Fixed!")

Fixed!


In [0]:
import sys
mods_to_remove = [k for k in sys.modules if 'bronze' in k]
for m in mods_to_remove:
    del sys.modules[m]
print("Cache cleared!")

Cache cleared!


In [0]:
with open('/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src/bronze/watermark.py', 'r') as f:
    print(f.read())

"""
loader.py
---------
Bronze loader — orchestrates the full source-to-bronze pipeline for one job.

Flow per file:
  1. Discover files under source URI
  2. Skip already-processed files (watermark check)
  3. Detect format
  4. Read via source-specific reader (ADLS/S3/GCS)
  5. Attach 8 metadata columns
  6. Append clean rows to bronze_<dataset>
  7. Append corrupt rows to bronze_corrupt
  8. Update watermark to bronze_loaded
"""

from __future__ import annotations
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from bronze.config.models import JobConfig
from bronze.discovery.file_lister import list_files, DiscoveredFile
from bronze.discovery.format_detector import detect_format
from bronze.metadata import attach_metadata, generate_run_id, generate_batch_id
from bronze.watermark import load_watermark, update_watermark, is_already_processed


# ---------------------------------------------------------------------------
# URI scheme router — picks the right

In [0]:
watermark_code = """from __future__ import annotations
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

WATERMARK_SCHEMA = StructType([
    StructField("content_hash",        StringType(),    False),
    StructField("file_name",           StringType(),    False),
    StructField("status",              StringType(),    False),
    StructField("ingestion_timestamp", TimestampType(), True),
])

def load_watermark(spark: SparkSession, location: str) -> DataFrame:
    try:
        return spark.read.parquet(location)
    except Exception:
        return spark.createDataFrame([], schema=WATERMARK_SCHEMA)

def is_already_processed(watermark_df: DataFrame, content_hash: str) -> bool:
    if watermark_df.rdd.isEmpty():
        return False
    match = watermark_df.filter(
        (F.col("content_hash") == content_hash) &
        (F.col("status") == "bronze_loaded")
    )
    return match.count() > 0

def update_watermark(
    spark: SparkSession,
    location: str,
    content_hash: str,
    file_name: str,
    status: str = "bronze_loaded",
) -> None:
    new_row = spark.createDataFrame(
        [(content_hash, file_name, status, None)],
        schema=WATERMARK_SCHEMA,
    ).withColumn("ingestion_timestamp", F.current_timestamp())
    new_row.write.mode("append").parquet(location)
"""

path = '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src/bronze/watermark.py'

with open(path, 'w') as f:
    f.write(watermark_code)

# Verify
with open(path, 'r') as f:
    content = f.read()
    print(content[:200])
    print("...")
    print("load_watermark" in content, "is_already_processed" in content, "update_watermark" in content)

from __future__ import annotations
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampTy
...
True True True


In [0]:
loader_code = """from __future__ import annotations
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from bronze.config.models import JobConfig
from bronze.discovery.file_lister import list_files
from bronze.discovery.format_detector import detect_format
from bronze.metadata import attach_metadata, generate_run_id, generate_batch_id
from bronze.watermark import load_watermark, update_watermark, is_already_processed

def _get_reader(scheme: str):
    if scheme == "abfss":
        from bronze.readers.adls_reader import read_from_adls
        return read_from_adls
    elif scheme == "s3a":
        from bronze.readers.s3_reader import read_from_s3
        return read_from_s3
    elif scheme == "gs":
        from bronze.readers.gcs_reader import read_from_gcs
        return read_from_gcs
    elif scheme == "file":
        from bronze.readers.adls_reader import read_from_adls
        return read_from_adls
    else:
        raise ValueError(f"Unsupported URI scheme: {scheme}")

def run_bronze_job(
    spark: SparkSession,
    job: JobConfig,
    watermark_location: str,
) -> dict:
    run_id   = generate_run_id()
    batch_id = generate_batch_id(job.job_id, run_id)
    print(f"[Bronze] Starting job: {job.job_id}")
    print(f"[Bronze] run_id:   {run_id}")
    print(f"[Bronze] batch_id: {batch_id}")

    discovered = list_files(job.source.uri, pattern=job.source.pattern)
    print(f"[Bronze] Files discovered: {len(discovered)}")

    watermark_df = load_watermark(spark, watermark_location)

    stats = {
        "files_found"  : len(discovered),
        "files_skipped": 0,
        "files_loaded" : 0,
        "rows_loaded"  : 0,
        "corrupt_rows" : 0,
    }

    for file in discovered:
        if is_already_processed(watermark_df, file.content_hash):
            print(f"[Bronze] Skipping (already loaded): {file.file_name}")
            stats["files_skipped"] += 1
            continue

        print(f"[Bronze] Processing: {file.file_name}")

        try:
            fmt    = detect_format(file.uri, config_override=job.source.format)
            reader = _get_reader(file.scheme)
            clean_df, corrupt_df = reader(spark, file.uri, job)

            clean_df = attach_metadata(
                clean_df,
                batch_id        = batch_id,
                run_id          = run_id,
                source_file_name= file.file_name,
                source_file_hash= file.content_hash,
            )

            if corrupt_df.count() > 0:
                corrupt_df = (
                    corrupt_df
                    .withColumn("_source_file_name",    F.lit(file.file_name))
                    .withColumn("_source_file_hash",    F.lit(file.content_hash))
                    .withColumn("_batch_id",            F.lit(batch_id))
                    .withColumn("_run_id",              F.lit(run_id))
                    .withColumn("_ingestion_timestamp", F.current_timestamp())
                )

            clean_count = clean_df.count()
            (
                clean_df.write
                .mode("append")
                .partitionBy("_ingestion_date")
                .parquet(f"{job.bronze.location}")
            )

            corrupt_count = corrupt_df.count()
            if corrupt_count > 0:
                corrupt_location = job.bronze.location.rsplit("/", 1)[0] + "/bronze_corrupt/"
                corrupt_df.write.mode("append").parquet(corrupt_location)

            update_watermark(
                spark,
                watermark_location,
                file.content_hash,
                file.file_name,
                status="bronze_loaded",
            )

            stats["files_loaded"]  += 1
            stats["rows_loaded"]   += clean_count
            stats["corrupt_rows"]  += corrupt_count
            print(f"[Bronze] Loaded: {file.file_name} — {clean_count} rows, {corrupt_count} corrupt")

        except Exception as exc:
            print(f"[Bronze] ERROR on {file.file_name}: {exc}")
            raise

    print(f"[Bronze] Job complete: {stats}")
    return stats
"""

path = '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src/bronze/loader.py'

with open(path, 'w') as f:
    f.write(loader_code)

print("loader.py written!")

loader.py written!


In [0]:
import sys
mods_to_remove = [k for k in sys.modules if 'bronze' in k]
for m in mods_to_remove:
    del sys.modules[m]
print("Cache cleared!")

Cache cleared!


In [0]:
import os
os.environ["AZURE_STORAGE_ACCOUNT_NAME"] = "medelliondataarcs"
os.environ["AZURE_STORAGE_ACCOUNT_KEY"] = "<your-storage-key>"

In [0]:
adls_lister = """from __future__ import annotations
import hashlib
import fnmatch
from dataclasses import dataclass
from pathlib import Path
from urllib.parse import urlparse


@dataclass
class DiscoveredFile:
    uri: str
    scheme: str
    file_name: str
    content_hash: str
    size_bytes: int
    detected_format: str | None = None


def _parse_scheme(uri: str) -> str:
    parsed = urlparse(uri)
    scheme = parsed.scheme.lower()
    if scheme in ("abfss", "s3a", "gs"):
        return scheme
    return "file"


def _sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _list_local(uri: str, pattern: str) -> list[tuple[str, bytes, int]]:
    path_str = uri.replace("file://", "")
    base = Path(path_str).resolve()
    if not base.exists():
        raise FileNotFoundError(f"Local path does not exist: {base}")
    results = []
    files = base.rglob("*") if "**" in pattern else base.glob(pattern)
    for f in files:
        if f.is_file():
            data = f.read_bytes()
            results.append((f.resolve().as_uri(), data, len(data)))
    return results


def _list_adls(uri: str, pattern: str) -> list[tuple[str, bytes, int]]:
    from azure.storage.filedatalake import DataLakeServiceClient
    from azure.storage.filedatalake import StorageStreamDownloader

    parsed = urlparse(uri)
    container = parsed.username
    account_host = parsed.hostname
    account_name = account_host.split(".")[0]
    prefix = parsed.path.lstrip("/")

    import os
    account_key = os.environ.get("AZURE_STORAGE_ACCOUNT_KEY")

    account_url = f"https://{account_host}"

    if account_key:
        service = DataLakeServiceClient(
            account_url=account_url,
            credential=account_key
        )
    else:
        from azure.identity import DefaultAzureCredential
        service = DataLakeServiceClient(
            account_url=account_url,
            credential=DefaultAzureCredential()
        )

    fs_client = service.get_file_system_client(container)
    results = []
    paths = fs_client.get_paths(path=prefix or "/", recursive=True)
    for p in paths:
        if not p.is_directory:
            fname = p.name.split("/")[-1]
            if fnmatch.fnmatch(fname, pattern):
                file_client = fs_client.get_file_client(p.name)
                download = file_client.download_file()
                data = download.readall()
                full_uri = f"abfss://{container}@{account_host}/{p.name}"
                results.append((full_uri, data, len(data)))
    return results


def _list_s3(uri: str, pattern: str) -> list[tuple[str, bytes, int]]:
    import boto3
    parsed = urlparse(uri)
    bucket = parsed.netloc
    prefix = parsed.path.lstrip("/")
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
    results = []
    for page in pages:
        for obj in page.get("Contents", []):
            key = obj["Key"]
            fname = key.split("/")[-1]
            if fnmatch.fnmatch(fname, pattern):
                response = s3.get_object(Bucket=bucket, Key=key)
                data = response["Body"].read()
                full_uri = f"s3a://{bucket}/{key}"
                results.append((full_uri, data, len(data)))
    return results


def _list_gcs(uri: str, pattern: str) -> list[tuple[str, bytes, int]]:
    from google.cloud import storage as gcs_storage
    parsed = urlparse(uri)
    bucket_name = parsed.netloc
    prefix = parsed.path.lstrip("/")
    client = gcs_storage.Client()
    blobs = client.list_blobs(bucket_name, prefix=prefix)
    results = []
    for blob in blobs:
        fname = blob.name.split("/")[-1]
        if fnmatch.fnmatch(fname, pattern):
            data = blob.download_as_bytes()
            full_uri = f"gs://{bucket_name}/{blob.name}"
            results.append((full_uri, data, len(data)))
    return results


_SCHEME_HANDLERS = {
    "abfss": _list_adls,
    "s3a"  : _list_s3,
    "gs"   : _list_gcs,
    "file" : _list_local,
}


def list_files(uri: str, pattern: str = "*") -> list[DiscoveredFile]:
    scheme = _parse_scheme(uri)
    if scheme not in _SCHEME_HANDLERS:
        raise ValueError(f"Unsupported URI scheme '{scheme}' in: {uri}")
    handler = _SCHEME_HANDLERS[scheme]
    raw_files = handler(uri, pattern)
    discovered = []
    for full_uri, data, size in raw_files:
        fname = full_uri.rstrip("/").split("/")[-1]
        discovered.append(
            DiscoveredFile(
                uri=full_uri,
                scheme=scheme,
                file_name=fname,
                content_hash=_sha256(data),
                size_bytes=size,
            )
        )
    return discovered
"""

path = '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src/bronze/discovery/file_lister.py'
with open(path, 'w') as f:
    f.write(adls_lister)
print("file_lister.py updated!")

file_lister.py updated!


In [0]:
import os
os.environ["AZURE_STORAGE_ACCOUNT_KEY"] = "<your-storage-key>"

import sys
mods_to_remove = [k for k in sys.modules if 'bronze' in k]
for m in mods_to_remove:
    del sys.modules[m]

sys.path.insert(0, '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src')

from bronze.loader import run_bronze_job
from bronze.config import YamlConfigStore

spark.conf.set(
    "fs.azure.account.key.medelliondataarcs.dfs.core.windows.net",
    "<your-storage-key>"
)

store = YamlConfigStore('/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/configs/example.yaml')
job = store.get_job("customers_raw")

watermark_location = "abfss://bronze@medelliondataarcs.dfs.core.windows.net/watermark/"

stats = run_bronze_job(spark, job, watermark_location)
print("Stats:", stats)

[Bronze] Starting job: customers_raw
[Bronze] run_id:   ddf3e73a-c638-4216-9219-a91847f9f0de
[Bronze] batch_id: customers_raw__ddf3e73a
[Bronze] Files discovered: 1
[Bronze] Processing: sample1.csv
[Bronze] Loaded: sample1.csv — 2 rows, 0 corrupt
[Bronze] Job complete: {'files_found': 1, 'files_skipped': 0, 'files_loaded': 1, 'rows_loaded': 2, 'corrupt_rows': 0}
Stats: {'files_found': 1, 'files_skipped': 0, 'files_loaded': 1, 'rows_loaded': 2, 'corrupt_rows': 0}


In [0]:
stats = run_bronze_job(spark, job, watermark_location)
print("Stats:", stats)

[Bronze] Starting job: customers_raw
[Bronze] run_id:   f70e321e-4219-4bae-8a5b-619839d81330
[Bronze] batch_id: customers_raw__f70e321e
[Bronze] Files discovered: 1
[Bronze] Skipping (already loaded): sample1.csv
[Bronze] Job complete: {'files_found': 1, 'files_skipped': 1, 'files_loaded': 0, 'rows_loaded': 0, 'corrupt_rows': 0}
Stats: {'files_found': 1, 'files_skipped': 1, 'files_loaded': 0, 'rows_loaded': 0, 'corrupt_rows': 0}


In [0]:
# Read the bronze table back and show it
df = spark.read.parquet("abfss://bronze@medelliondataarcs.dfs.core.windows.net/customers/")
print(f"Total rows in bronze table: {df.count()}")
df.show()

Total rows in bronze table: 2
+---+-----+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------------+
| id| name|           _batch_id|             _run_id|_source_file_name|   _source_file_hash|_ingestion_timestamp|_record_seq|    _source_row_uuid|_ingestion_date|
+---+-----+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------------+
|  1|Alice|customers_raw__dd...|ddf3e73a-c638-421...|      sample1.csv|b5c2910951a29cc50...|2026-04-29 12:43:...|          0|943f836d-e2d8-44b...|     2026-04-29|
|  2|  Bob|customers_raw__dd...|ddf3e73a-c638-421...|      sample1.csv|b5c2910951a29cc50...|2026-04-29 12:43:...|          1|4d084fcb-4420-4d0...|     2026-04-29|
+---+-----+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------

In [0]:
import os
os.environ["AZURE_STORAGE_ACCOUNT_KEY"] = "<your-storage-key>"

import sys
mods_to_remove = [k for k in sys.modules if 'bronze' in k]
for m in mods_to_remove:
    del sys.modules[m]

sys.path.insert(0, '/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/src')

from bronze.loader import run_bronze_job
from bronze.config import YamlConfigStore

spark.conf.set(
    "fs.azure.account.key.medelliondataarcs.dfs.core.windows.net",
    "<your-storage-key>"
)

store = YamlConfigStore('/Workspace/Users/kinjal.kanjilal.aiml25@heritageit.edu.in/Building_a_reusable_medellion_data_architecture/configs/example.yaml')
job = store.get_job("customers_raw")

watermark_location = "abfss://bronze@medelliondataarcs.dfs.core.windows.net/watermark/"

stats = run_bronze_job(spark, job, watermark_location)
print("Stats:", stats)

[Bronze] Starting job: customers_raw
[Bronze] run_id:   68faf15a-52c8-4186-8c31-2588abc20a20
[Bronze] batch_id: customers_raw__68faf15a
[Bronze] Files discovered: 3
[Bronze] Skipping (already loaded): sample1.csv
[Bronze] Processing: sample2.json
[Bronze] Loaded: sample2.json — 1 rows, 0 corrupt
[Bronze] Processing: sample3.csv
[Bronze] Loaded: sample3.csv — 1 rows, 0 corrupt
[Bronze] Job complete: {'files_found': 3, 'files_skipped': 1, 'files_loaded': 2, 'rows_loaded': 2, 'corrupt_rows': 0}
Stats: {'files_found': 3, 'files_skipped': 1, 'files_loaded': 2, 'rows_loaded': 2, 'corrupt_rows': 0}


In [0]:
# Read the full bronze table
df = spark.read.parquet("abfss://bronze@medelliondataarcs.dfs.core.windows.net/customers/")

print(f"Total rows in bronze table: {df.count()}")
print(f"Columns: {df.columns}")
print()
df.show()

Total rows in bronze table: 4
Columns: ['id', 'name', '_batch_id', '_run_id', '_source_file_name', '_source_file_hash', '_ingestion_timestamp', '_record_seq', '_source_row_uuid', '_ingestion_date']

+---+-------+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------------+
| id|   name|           _batch_id|             _run_id|_source_file_name|   _source_file_hash|_ingestion_timestamp|_record_seq|    _source_row_uuid|_ingestion_date|
+---+-------+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------------+
|  1|  Alice|customers_raw__dd...|ddf3e73a-c638-421...|      sample1.csv|b5c2910951a29cc50...|2026-04-29 12:43:...|          0|943f836d-e2d8-44b...|     2026-04-29|
|  2|    Bob|customers_raw__dd...|ddf3e73a-c638-421...|      sample1.csv|b5c2910951a29cc50...|2026-04-29 12:43:...|          1|4d084fcb-4420-

In [0]:
df.filter(df._source_file_name == "sample2.json").show()


+---+-----+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------------+
| id| name|           _batch_id|             _run_id|_source_file_name|   _source_file_hash|_ingestion_timestamp|_record_seq|    _source_row_uuid|_ingestion_date|
+---+-----+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------------+
|  1|Alice|customers_raw__68...|68faf15a-52c8-418...|     sample2.json|91ba5021aa1e19a5c...|2026-04-29 13:13:...|          0|91e3593f-2007-484...|     2026-04-29|
+---+-----+--------------------+--------------------+-----------------+--------------------+--------------------+-----------+--------------------+---------------+

